<a href="https://colab.research.google.com/github/liqq1024/GenAI_LLMs_2026Fall/blob/main/colab-notebooks/Week3_SelfAttention_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: Self-Attention and Transformers

## Live Coding Activity (Google Colab)

This activity connects the Week 2 pipeline—text to tokens, token IDs, and embeddings—to the self-attention mechanism used by Transformers.

### Objectives

- Implement scaled dot-product attention from scratch.
- Use PyTorch tensors for matrix operations.
- Visualize attention weights.
- Compare embeddings before and after attention.
- Verify the calculation with PyTorch's MultiheadAttention.

> This notebook uses a toy example. Its vectors are not trained language-model representations; they make the attention calculation visible.


## Setup

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Section 1: Review Embeddings from Week 2

A token ID is an index. An embedding layer maps each ID to a learned dense vector. Self-attention then lets each token update its vector by gathering information from other tokens in the same sequence.


In [ ]:
tokens = ["plants", "need", "sunlight", "and", "water"]
word_to_id = {word: index for index, word in enumerate(tokens)}
token_ids = torch.tensor([word_to_id[word] for word in tokens])

embedding_layer = nn.Embedding(num_embeddings=len(tokens), embedding_dim=8)
week2_embeddings = embedding_layer(token_ids)

print("Tokens:", tokens)
print("Token IDs:", token_ids.tolist())
print("Embedding shape:", tuple(week2_embeddings.shape))

# fill the code 1: print out the embedding for "plants"
print("Embedding for plants:", )

**Quick discussion:** Why can a neural network perform calculations with an embedding vector but not with a raw word? Why are IDs only labels rather than meaningful quantities?

## Section 2: Create Toy Embeddings

The input is a matrix with one row per token and eight features per embedding.


In [ ]:
sequence_length = 5
embedding_dim = 8
embeddings = torch.rand(sequence_length, embedding_dim, device=device)

print("embeddings.shape:", tuple(embeddings.shape))
print("Rows correspond to:", tokens)
print(embeddings.cpu())

## Section 3: Create Query, Key, and Value (Q, K, V)

Q, K, and V are different learned projections of the input embeddings:

- Query: what a token is looking for.
- Key: what a token offers for matching.
- Value: information a token contributes when attended to.

They are learned representations, not fixed linguistic categories.


In [ ]:
attention_dim = 8
query_layer = nn.Linear(embedding_dim, attention_dim, bias=False).to(device)
key_layer = nn.Linear(embedding_dim, attention_dim, bias=False).to(device)
value_layer = nn.Linear(embedding_dim, attention_dim, bias=False).to(device)

Q = query_layer(embeddings)
K = key_layer(embeddings)
V = value_layer(embeddings)

print("Q shape:", tuple(Q.shape))
print("K shape:", tuple(K.shape))
print("V shape:", tuple(V.shape))

## Section 4: Compute Attention Scores

The matrix product Q multiplied by K transpose compares every query with every key. A row represents the query token; a column represents the candidate key token.


In [ ]:
# fill the code 2: caculate the attention scores
attention_scores =
print("attention_scores.shape:", tuple(attention_scores.shape))
print("Raw QK transpose scores:")
print(attention_scores.cpu())

**Predict:** After softmax, what should the sum of every row in the attention matrix be?

## Section 5: Scale and Normalize Scores

Scaled dot-product attention uses:

Attention(Q, K, V) = softmax(QK transpose / sqrt(d_k)) V

Scaling prevents large dot products from making softmax too peaked. Softmax converts each row to weights that sum to one.


In [ ]:
scaled_scores = attention_scores / math.sqrt(attention_dim)
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("attention_weights.shape:", tuple(attention_weights.shape))
print(attention_weights.cpu())
print("Row sums:", attention_weights.sum(dim=-1).cpu())

## Section 6: Aggregate the Values

Each output embedding is a weighted combination of all value vectors. It is a context-aware representation of the original token.


In [ ]:
# fill the code 3: caculate the attention output
attention_output =
print("attention_output.shape:", tuple(attention_output.shape))

for index, token in enumerate(tokens):
    print("\nToken:", token)
    print("Original embedding:", embeddings[index].detach().cpu())
    print("After attention:  ", attention_output[index].detach().cpu())

**Discuss:** Is attention permanent memory? Explain why it is better described as weighted aggregation for the current calculation.

## Section 7: Visualize the Attention Matrix

Read a row from left to right. For example, the row for plants shows how much plants attends to every token, including itself.


In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(attention_weights.detach().cpu(), cmap="viridis", aspect="auto")
plt.colorbar(label="Attention weight")
plt.xticks(range(sequence_length), tokens, rotation=30, ha="right")
plt.yticks(range(sequence_length), tokens)
plt.xlabel("Key / value token")
plt.ylabel("Query token")
plt.title("Single-Head Self-Attention Weights")

for i in range(sequence_length):
    for j in range(sequence_length):
        weight = attention_weights[i, j].item()
        color = "white" if weight < 0.35 else "black"
        plt.text(j, i, f"{weight:.2f}", ha="center", va="center", color=color)

plt.tight_layout()
plt.show()

for query_token, key_id in zip(tokens, attention_weights.argmax(dim=-1).cpu().tolist()):
    print(f"{query_token:>9} attends most to: {tokens[key_id]}")

## Section 8: Compare with PyTorch MultiheadAttention

We use one attention head and copy the manual Q, K, and V projection weights into PyTorch's module. The output projection is set to the identity matrix. The manual and module outputs should match except for tiny floating-point differences.


In [ ]:
pytorch_attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=1,
    bias=False,
    batch_first=True,
    dropout=0.0
).to(device)

with torch.no_grad():
    pytorch_attention.in_proj_weight.copy_(
        torch.cat([query_layer.weight, key_layer.weight, value_layer.weight], dim=0)
    )
    pytorch_attention.out_proj.weight.copy_(torch.eye(embedding_dim, device=device))

attention_input = embeddings.unsqueeze(0)
module_output, module_weights = pytorch_attention(
    attention_input, attention_input, attention_input,
    need_weights=True, average_attn_weights=False
)
module_output = module_output.squeeze(0)
module_weights = module_weights.squeeze(0).squeeze(0)

print("Manual output shape:", tuple(attention_output.shape))
print("Module output shape:", tuple(module_output.shape))
print("Outputs match:", torch.allclose(attention_output, module_output, atol=1e-6))
print("Weights match:", torch.allclose(attention_weights, module_weights, atol=1e-6))
print("Maximum output difference:", (attention_output - module_output).abs().max().item())

## Exit Ticket

Answer in a Markdown cell:

1. What are the shapes of Q, K, V, QK transpose, attention weights, and the final attention output?
2. What does one row of the attention-weight matrix represent?
3. Why do we divide by square root of d_k before softmax?
4. How is an attention output embedding different from the original embedding?
5. Why does a Transformer need positional encoding?

## Instructor Notes: Common Misconceptions

- **Attention is not memory.** It is a weighted aggregation mechanism.
- **Queries, keys, and values are learned representations**, not fixed linguistic concepts.
- **Multi-head attention does not simply repeat the same computation.** Each head learns separate projections and may capture different relationships.
- **Self-attention alone is permutation invariant.** Positional encoding supplies token-order information.

### Optional extension

Change the module to two heads while keeping the embedding dimension at eight. Inspect the returned attention-weight shape and explain why each head has its own attention pattern.

## Key takeaway

Self-attention transforms each token embedding into a context-aware vector by comparing queries and keys, normalizing scores into weights, and aggregating values.
